In [ ]:
%pip install -q sagemaker boto3 pandas numpy matplotlib

# Week 2 Friday -- Advanced SageMaker Inference Patterns + Model Monitoring

Monday you deployed a single real-time endpoint. Today you learn the full inference spectrum: serverless (scale to zero), asynchronous (large payloads), batch transform (bulk scoring), and multi-model endpoints (shared infrastructure). Then you set up model monitoring -- because a deployed model that degrades silently is worse than no model at all.

By the end of this notebook you will have:

1. **Deployed** Thursday's XGBoost model using serverless, async, and batch transform patterns.
2. **Deployed** a real-time endpoint with **data capture** enabled for monitoring.
3. **Created a baseline** from training data and **scheduled** data quality monitoring.
4. **Simulated data drift** and understood how Model Monitor flags violations.
5. Designed an **EventBridge automation** pattern to close the drift-to-retrain loop.

| Block | Content |
|-------|---------|
| Stage 1 | Inference Patterns: Decision Matrix, Serverless, Async, Batch Transform |
| Stage 2 | Model Monitoring Setup: Data Capture, Baselines, Quality Schedules |
| Stage 3 | Drift Detection and Automation |

**Pairing callouts:**

- **Thursday (W2):** We use Thursday's best XGBoost model artifact (from HPO) as the model we deploy with different inference patterns.
- **Monday (W2):** Monday deployed a real-time endpoint and you invoked it. Today we see real-time is just one of five inference options. The data capture we enable today adds the monitoring layer Monday's deployment lacked.

## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region.

In [ ]:
import boto3
import sagemaker
from datetime import datetime

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

sm_client = boto3.client("sagemaker")
s3 = boto3.client("s3")

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Region:  {region}")
print(f"Bucket:  s3://{bucket}")
print(f"Role:    ...{role[-30:]}")

In [ ]:
import json
import time

runtime_client = boto3.client("sagemaker-runtime")

## Verify Thursday's XGBoost Model Artifact

We need the best XGBoost model artifact from Thursday's HPO job. The cell below searches S3 for it. If it is missing, the fallback cell retrains a quick XGBoost model.

> **What is happening:** We list S3 objects under the `fraudshield/` prefix to find an XGBoost `model.tar.gz`. We also verify the training/validation data is available.

In [ ]:
xgb_model_uri = None
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("model.tar.gz") and ("xgb" in key.lower() or "hpo" in key.lower() or "tuning" in key.lower()):
            xgb_model_uri = f"s3://{bucket}/{key}"

if not xgb_model_uri:
    for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/output/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("model.tar.gz"):
                xgb_model_uri = f"s3://{bucket}/{obj['Key']}"

train_exists = False
val_exists = False
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/train/train.csv")
    train_exists = True
except Exception:
    pass
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/validation/validation.csv")
    val_exists = True
except Exception:
    pass

print(f"XGBoost model artifact: {xgb_model_uri or 'NOT FOUND'}")
print(f"Training CSV:           {'Found' if train_exists else 'NOT FOUND'}")
print(f"Validation CSV:         {'Found' if val_exists else 'NOT FOUND'}")

if xgb_model_uri and train_exists and val_exists:
    print("\nAll artifacts present. Proceed to Stage 1.")
else:
    print("\nMissing artifacts -- run the fallback cell below.")

### Fallback: Regenerate Data and Train XGBoost (only if needed)

If artifacts are missing, the cell below regenerates the synthetic fraud data and trains an XGBoost model using the SageMaker built-in algorithm. This takes approximately 5-7 minutes.

In [ ]:
# --- ONLY RUN THIS CELL IF ARTIFACTS ARE MISSING ---

import numpy as np
import pandas as pd
import os

if not xgb_model_uri or not train_exists or not val_exists:
    np.random.seed(42)
    n = 2000
    data = pd.DataFrame({
        "amount": np.random.exponential(500, n).round(2),
        "hour": np.random.randint(0, 24, n),
        "distance_from_home": np.random.exponential(50, n).round(2),
        "transaction_count_24h": np.random.poisson(5, n),
        "is_international": np.random.binomial(1, 0.1, n),
        "merchant_risk_score": np.random.uniform(0, 1, n).round(3),
    })
    data["target"] = (
        (data["amount"] > 800) & (data["hour"] < 6) | (data["merchant_risk_score"] > 0.85)
    ).astype(int)
    noise = np.random.random(n) < 0.08
    data["target"] = (data["target"] ^ noise.astype(int))

    train = data.iloc[:1600]
    val = data.iloc[1600:]

    train_xgb = pd.concat([train["target"], train.drop("target", axis=1)], axis=1)
    val_xgb = pd.concat([val["target"], val.drop("target", axis=1)], axis=1)
    train_xgb.to_csv("train_xgb.csv", index=False, header=False)
    val_xgb.to_csv("val_xgb.csv", index=False, header=False)

    train.to_csv("train.csv", index=False)
    val.to_csv("validation.csv", index=False)

    s3.upload_file("train.csv", bucket, f"{prefix}/data/train/train.csv")
    s3.upload_file("validation.csv", bucket, f"{prefix}/data/validation/validation.csv")
    s3.upload_file("train_xgb.csv", bucket, f"{prefix}/data/xgb-train/train.csv")
    s3.upload_file("val_xgb.csv", bucket, f"{prefix}/data/xgb-validation/validation.csv")
    train_exists = True
    val_exists = True

    from sagemaker import image_uris
    xgb_image = image_uris.retrieve("xgboost", region, version="1.7-1")

    xgb_estimator = sagemaker.estimator.Estimator(
        image_uri=xgb_image,
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        output_path=f"s3://{bucket}/{prefix}/xgb-output/",
        sagemaker_session=session,
    )
    xgb_estimator.set_hyperparameters(
        objective="binary:logistic",
        num_round=100,
        max_depth=5,
        eta=0.2,
        eval_metric="auc",
    )
    xgb_estimator.fit({
        "train": sagemaker.inputs.TrainingInput(
            f"s3://{bucket}/{prefix}/data/xgb-train/", content_type="text/csv"
        ),
        "validation": sagemaker.inputs.TrainingInput(
            f"s3://{bucket}/{prefix}/data/xgb-validation/", content_type="text/csv"
        ),
    }, wait=True, logs="All")

    xgb_model_uri = xgb_estimator.model_data
    print(f"\nTrained XGBoost model artifact: {xgb_model_uri}")
else:
    print("Artifacts already present -- skipping retrain.")

---
# STAGE 1 -- Inference Patterns

**Connecting to Thursday:** Thursday you trained an XGBoost model with hyperparameter optimization. The best model artifact sits in S3. Today we deploy that same model four different ways and learn when each pattern is appropriate.

**Connecting to Monday:** Monday you deployed a real-time endpoint and invoked it. Real-time is just one of five inference patterns. Today we see the full spectrum.

## Inference Decision Matrix

Before deploying, you need to choose the right inference pattern for your workload. This table is your decision guide.

| Dimension | Real-Time | Serverless | Async | Batch Transform | Multi-Model |
|-----------|-----------|------------|-------|-----------------|-------------|
| **Latency** | Low (ms) | Variable (cold start 1-2s) | Minutes (queued) | Minutes to hours | Low (ms, after model load) |
| **Traffic Pattern** | Steady, high volume | Bursty, unpredictable | Large/slow jobs | One-time or scheduled bulk | Many models, moderate traffic each |
| **Max Payload** | 6 MB | 4 MB | 1 GB | Unlimited (S3-based) | 6 MB |
| **Scale to Zero** | No | Yes | No (but can with custom policy) | N/A (ephemeral) | No |
| **GPU Support** | Yes | No | Yes | Yes | Yes |
| **Billing** | Per instance-hour | Per ms of compute | Per instance-hour | Per instance-hour (job duration) | Per instance-hour (shared) |

**Discussion:** Look at FraudShield's workload. Card transactions are steady at 500 TPS during business hours, near-zero between 2 AM and 6 AM. Which pattern fits? What if you also need to score a month's worth of historical transactions every quarter?

## Inference Decision Flowchart

Use this flowchart when a stakeholder says "deploy this model":

```
Is it a one-time bulk scoring job?
  YES --> Batch Transform
  NO  --> Is the traffic pattern bursty with long idle periods?
            YES --> Does the payload exceed 4 MB?
                      YES --> Async Inference
                      NO  --> Serverless Inference
            NO  --> Do you need to serve many models on shared infrastructure?
                      YES --> Multi-Model Endpoint
                      NO  --> Real-Time Endpoint
```

The first question is not "which instance type" -- it is "what does the traffic look like?"

## STEP 1 -- Create a Model Object

Before deploying with different inference patterns, we need a SageMaker `Model` object. This links the model artifact in S3 to the inference container image.

This is the same three-object pattern from Monday (Model -> Endpoint Configuration -> Endpoint), but today we use different endpoint configurations for each inference pattern.

> **What is happening:** We retrieve the XGBoost container image URI for inference and create a `Model` object pointing to Thursday's trained artifact.

In [ ]:
from sagemaker import image_uris
from sagemaker.model import Model

xgb_image = image_uris.retrieve("xgboost", region, version="1.7-1")

xgb_model = Model(
    image_uri=xgb_image,
    model_data=xgb_model_uri,
    role=role,
    sagemaker_session=session,
    name=f"fraudshield-xgb-{TIMESTAMP}",
)

print(f"Model name:     {xgb_model.name}")
print(f"Image URI:      {xgb_image}")
print(f"Model artifact: {xgb_model_uri}")

## STEP 2 -- Serverless Inference

Serverless inference is the "Lambda for models." You configure memory and max concurrency. SageMaker provisions compute on demand and **scales to zero** when idle. You pay only for the milliseconds your inference code runs.

**Key configuration:**
- `memory_size_in_mb`: 1024, 2048, 3072, 4096, 5120, or 6144
- `max_concurrency`: 1 to 200 (simultaneous invocations)

**Trade-offs:**
- Cold start: first invocation after idle takes 1-2 seconds
- No GPU support -- CPU only
- 4 MB max payload (vs 6 MB for real-time)
- Great for: dev/test, bursty traffic, internal tools, low-traffic models

> **What is happening:** We deploy the XGBoost model with a `ServerlessInferenceConfig`. The endpoint will have no running instances when idle -- compute is provisioned only when a request arrives.

In [ ]:
from sagemaker.serverless import ServerlessInferenceConfig

SERVERLESS_ENDPOINT = f"fraudshield-serverless-{TIMESTAMP}"

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)

print(f"Deploying serverless endpoint: {SERVERLESS_ENDPOINT}")
print("This takes 2-4 minutes.")

serverless_predictor = xgb_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=SERVERLESS_ENDPOINT,
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer(),
)

print(f"\nServerless endpoint in service: {SERVERLESS_ENDPOINT}")

### Invoke the Serverless Endpoint -- Observe Cold Start

The first invocation after idle triggers a cold start (container provisioning). Run the cell below, then run it again immediately to see the latency difference.

> **What is happening:** We send a single row of feature data to the serverless endpoint. The first call provisions the container (slow). The second call reuses the warm container (fast).

In [ ]:
import pandas as pd
import time

s3.download_file(bucket, f"{prefix}/data/validation/validation.csv", "validation.csv")
val_df = pd.read_csv("validation.csv")
features = val_df.drop("target", axis=1)

sample_row = features.iloc[0:1].values.tolist()[0]
sample_csv = ",".join(str(v) for v in sample_row)

for i in range(2):
    start = time.time()
    response = runtime_client.invoke_endpoint(
        EndpointName=SERVERLESS_ENDPOINT,
        ContentType="text/csv",
        Body=sample_csv,
    )
    elapsed = time.time() - start
    result = response["Body"].read().decode("utf-8").strip()
    label = "COLD START" if i == 0 else "WARM"
    print(f"Invocation {i+1} ({label}): prediction={result}, latency={elapsed:.3f}s")

print("\nNotice the latency difference. Cold start is the trade-off for scale-to-zero billing.")

## STEP 3 -- Asynchronous Inference

Async inference handles payloads up to **1 GB** and requests that take up to **15 minutes** to process. The client sends the request, gets back a token immediately, and the result is written to S3 when processing completes.

**How it works:**
1. Client sends inference request
2. SageMaker places the request in an internal queue
3. Request is processed when capacity is available
4. Result is written to the configured S3 output path
5. Client polls S3 for the result (or receives an SNS notification)

**Use cases:** Large document processing, video analysis, complex feature engineering at inference time, any workload where the client can tolerate a delay.

> **What is happening:** We create a new Model object (because the previous one was consumed by the serverless deploy) and deploy with an `AsyncInferenceConfig` that specifies where results are written in S3.

In [ ]:
from sagemaker.async_inference import AsyncInferenceConfig

ASYNC_ENDPOINT = f"fraudshield-async-{TIMESTAMP}"
async_output_s3 = f"s3://{bucket}/{prefix}/async-output/"

xgb_model_async = Model(
    image_uri=xgb_image,
    model_data=xgb_model_uri,
    role=role,
    sagemaker_session=session,
    name=f"fraudshield-xgb-async-{TIMESTAMP}",
)

async_config = AsyncInferenceConfig(
    output_path=async_output_s3,
)

print(f"Deploying async endpoint: {ASYNC_ENDPOINT}")
print(f"Results will be written to: {async_output_s3}")
print("This takes 3-5 minutes.")

async_predictor = xgb_model_async.deploy(
    instance_type="ml.m5.xlarge",
    initial_instance_count=1,
    async_inference_config=async_config,
    endpoint_name=ASYNC_ENDPOINT,
)

print(f"\nAsync endpoint in service: {ASYNC_ENDPOINT}")

### Invoke the Async Endpoint

Unlike real-time and serverless endpoints, async invocation does not return the prediction immediately. Instead, you get a response containing the S3 output location. The prediction is written there once processing completes.

> **What is happening:** We upload a small CSV payload to S3, then invoke the async endpoint using `invoke_endpoint_async`. The response contains the output location. We poll S3 until the result appears.

In [ ]:
sample_rows = features.iloc[:5]
sample_payload = sample_rows.to_csv(index=False, header=False)

async_input_key = f"{prefix}/async-input/sample-{TIMESTAMP}.csv"
s3.put_object(Bucket=bucket, Key=async_input_key, Body=sample_payload)
async_input_s3 = f"s3://{bucket}/{async_input_key}"

response = runtime_client.invoke_endpoint_async(
    EndpointName=ASYNC_ENDPOINT,
    InputLocation=async_input_s3,
    ContentType="text/csv",
)

output_location = response["OutputLocation"]
print(f"Request submitted. Output will be at: {output_location}")

from urllib.parse import urlparse
parsed = urlparse(output_location)
output_bucket = parsed.netloc
output_key = parsed.path.lstrip("/")

print("Polling for result...")
for attempt in range(30):
    try:
        result_obj = s3.get_object(Bucket=output_bucket, Key=output_key)
        result_body = result_obj["Body"].read().decode("utf-8").strip()
        print(f"\nAsync result received after {attempt + 1} poll(s):")
        print(f"Predictions: {result_body}")
        break
    except s3.exceptions.NoSuchKey:
        time.sleep(5)
else:
    print("Result not ready after 150 seconds. Check S3 manually.")

## STEP 4 -- Batch Transform

Batch transform is **not an endpoint**. It is an ephemeral job:

1. SageMaker spins up instances
2. Reads your input data from S3
3. Runs inference on every record
4. Writes the output to S3
5. Tears down the instances

No persistent infrastructure. No billing after the job ends.

**Use cases:** Monthly scoring runs, data backfills, regulatory reporting, any scenario where you have all the data upfront and do not need real-time responses.

**Key parameters:**
- `strategy`: `MultiRecord` (batch records per request) or `SingleRecord`
- `split_type`: `Line` (split input by newline), `None` (send whole file)
- `join_source`: `Input` to append predictions alongside original data

> **What is happening:** We create a Transformer from a new Model object and run batch inference on the validation CSV. SageMaker reads each line, sends it to the model, and writes predictions to S3.

In [ ]:
val_no_target = features.copy()
val_no_target.to_csv("val_features_only.csv", index=False, header=False)
val_input_key = f"{prefix}/batch-input/val_features.csv"
s3.upload_file("val_features_only.csv", bucket, val_input_key)
batch_input_s3 = f"s3://{bucket}/{val_input_key}"
batch_output_s3 = f"s3://{bucket}/{prefix}/batch-output/{TIMESTAMP}/"

xgb_model_batch = Model(
    image_uri=xgb_image,
    model_data=xgb_model_uri,
    role=role,
    sagemaker_session=session,
    name=f"fraudshield-xgb-batch-{TIMESTAMP}",
)

transformer = xgb_model_batch.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=batch_output_s3,
    strategy="MultiRecord",
    accept="text/csv",
)

print(f"Starting batch transform job...")
print(f"Input:  {batch_input_s3}")
print(f"Output: {batch_output_s3}")

transformer.transform(
    data=batch_input_s3,
    content_type="text/csv",
    split_type="Line",
    wait=True,
    logs=True,
)

print(f"\nBatch transform complete. Results at: {batch_output_s3}")

## Multi-Model Endpoints -- Conceptual Overview

**Problem:** FraudShield has 50 regional fraud models -- one per state. Deploying 50 separate endpoints is expensive and operationally complex.

**Solution:** A **multi-model endpoint** hosts all 50 models on a single endpoint. SageMaker dynamically loads and unloads models based on traffic.

**How it works:**
- All models share the same container image and instance
- Models are stored in S3 and loaded into memory on demand
- When memory is full, least-recently-used models are evicted
- The `TargetModel` parameter in the invoke call specifies which model to use

```python
# Conceptual -- how you would invoke a multi-model endpoint
response = runtime_client.invoke_endpoint(
    EndpointName="multi-model-endpoint",
    TargetModel="california-fraud-model.tar.gz",  # <-- selects the model
    ContentType="text/csv",
    Body=payload,
)
```

**Multi-model vs multi-container endpoints:**

| Feature | Multi-Model | Multi-Container |
|---------|------------|----------------|
| Models share | Same container, same instance | Different containers, same instance |
| Use case | Many similar models (same framework) | Pipeline stages (preprocessing + inference) |
| Model selection | `TargetModel` parameter | `TargetContainerHostname` parameter |

We do not deploy a multi-model endpoint today because it requires multiple model artifacts. The concept is important for production architectures and the exam.

**Discussion:** What is the trade-off of multi-model endpoints vs separate endpoints? (Shared infrastructure is cheaper but introduces latency for model loading and potential resource contention.)

## Cleanup Stage 1 -- Delete Inference Endpoints

Delete the serverless and async endpoints before proceeding. We will deploy a fresh real-time endpoint with data capture in Stage 2.

> **What is happening:** We delete the serverless and async endpoints, their configurations, and their model objects. The batch transform job already tore down its own resources.

In [ ]:
def safe_delete_endpoint(endpoint_name):
    """Delete endpoint, endpoint config, and print status."""
    try:
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print(f"  Deleted endpoint: {endpoint_name}")
    except Exception as e:
        print(f"  Endpoint {endpoint_name}: {e}")
    try:
        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
        print(f"  Deleted endpoint config: {endpoint_name}")
    except Exception as e:
        print(f"  Endpoint config {endpoint_name}: {e}")

def safe_delete_model(model_name):
    """Delete a SageMaker model object."""
    try:
        sm_client.delete_model(ModelName=model_name)
        print(f"  Deleted model: {model_name}")
    except Exception as e:
        print(f"  Model {model_name}: {e}")

print("Cleaning up Stage 1 resources...")
safe_delete_endpoint(SERVERLESS_ENDPOINT)
safe_delete_endpoint(ASYNC_ENDPOINT)
safe_delete_model(f"fraudshield-xgb-{TIMESTAMP}")
safe_delete_model(f"fraudshield-xgb-async-{TIMESTAMP}")
safe_delete_model(f"fraudshield-xgb-batch-{TIMESTAMP}")
print("\nStage 1 cleanup complete.")

---
# STAGE 2 -- Model Monitoring Setup

**Why monitoring matters:** Fraud patterns change. Data distributions shift. A model that was 95% accurate last month might be 70% accurate today -- and you will not know unless you monitor it.

Imagine FraudShield deploys the XGBoost model in January. It performs well because the training data matches live traffic. By March, a new type of fraud emerges -- cryptocurrency-related transactions with different feature distributions. The model has never seen these patterns. It starts misclassifying them as legitimate. Without monitoring, nobody notices until the quarterly review reveals millions in losses.

Model Monitor is the safety net.

## Model Monitor Architecture

The monitoring pipeline has four components:

```
Live Endpoint
    |
    v
Data Capture (logs requests + responses to S3)
    |
    v
Baseline (statistical profile of training data)
    |
    v
Monitoring Schedule (periodic job compares captured data to baseline)
    |
    v
Violations Report (JSON listing which features violated constraints)
```

- **Data Capture** is the raw material -- it logs every (or a sample of) request and response.
- **Baseline** is the reference -- statistical fingerprint of training data.
- **Monitoring Schedule** is the engine -- a recurring job that compares captured data to baseline.
- **Violations Report** is the output -- a JSON document listing which features drifted and by how much.

## STEP 5 -- Deploy with Data Capture Enabled

We deploy a fresh real-time endpoint, but this time we enable `DataCaptureConfig`. This tells SageMaker to log every request and response to S3 in JSON Lines format.

**Configuration:**
- `enable_capture=True`
- `sampling_percentage=100` (capture everything -- in production, use 10-20%)
- `capture_options`: capture both `Input` and `Output`
- `destination_s3_uri`: where the captured data lands

> **What is happening:** We create a new Model object and deploy it with DataCaptureConfig. Every inference request and response will be logged to S3 for later analysis.

In [ ]:
from sagemaker.model_monitor import DataCaptureConfig

MONITOR_ENDPOINT = f"fraudshield-monitor-{TIMESTAMP}"
capture_s3_uri = f"s3://{bucket}/{prefix}/data-capture/{MONITOR_ENDPOINT}"

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=capture_s3_uri,
    capture_options=["Input", "Output"],
)

xgb_model_monitor = Model(
    image_uri=xgb_image,
    model_data=xgb_model_uri,
    role=role,
    sagemaker_session=session,
    name=f"fraudshield-xgb-monitor-{TIMESTAMP}",
)

print(f"Deploying monitored endpoint: {MONITOR_ENDPOINT}")
print(f"Capture destination: {capture_s3_uri}")
print("This takes 3-5 minutes.")

monitor_predictor = xgb_model_monitor.deploy(
    instance_type="ml.m5.xlarge",
    initial_instance_count=1,
    endpoint_name=MONITOR_ENDPOINT,
    data_capture_config=data_capture_config,
)

print(f"\nMonitored endpoint in service: {MONITOR_ENDPOINT}")
print(f"Data capture writing to: {capture_s3_uri}")

## STEP 6 -- Create a Baseline from Training Data

The baseline is a statistical fingerprint of your training data. Model Monitor compares incoming data against this fingerprint. If the incoming data looks significantly different, it raises a violation.

The baselining job produces two files:

| File | Contents | Purpose |
|------|----------|--------|
| `statistics.json` | Per-feature: mean, std, min, max, quantiles, distribution type | Reference distribution |
| `constraints.json` | Per-feature: data type, completeness, allowed range, thresholds | Violation thresholds |

> **What is happening:** We run a SageMaker Processing job that reads the training CSV, computes statistics for every feature (mean, standard deviation, quantiles, distribution type), and generates constraint thresholds. This takes 3-5 minutes.

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

training_data_s3_uri = f"s3://{bucket}/{prefix}/data/train/train.csv"
baseline_s3_uri = f"s3://{bucket}/{prefix}/monitoring/baseline/{TIMESTAMP}"

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=5,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

print(f"Creating baseline from: {training_data_s3_uri}")
print(f"Baseline output: {baseline_s3_uri}")
print("This runs a Processing job (3-5 minutes)...")

monitor.suggest_baseline(
    baseline_dataset=training_data_s3_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_s3_uri,
    wait=True,
    logs=False,
)

print(f"\nBaseline created at: {baseline_s3_uri}")
print("Files generated: statistics.json, constraints.json")

### Inspect the Baseline

Let us look at what the baselining job produced. The statistics file contains per-feature statistical profiles, and the constraints file contains the thresholds that monitoring will enforce.

> **What is happening:** We download and display the baseline statistics and constraints. These are the reference values that all future monitoring comparisons will use.

In [ ]:
baseline_job = monitor.latest_baselining_job

print("=== Baseline Statistics (summary) ===")
try:
    stats = baseline_job.baseline_statistics().body_dict
    for feature in stats.get("features", []):
        name = feature.get("name", "unknown")
        num_stats = feature.get("numerical_statistics", {})
        if num_stats:
            mean = num_stats.get("mean", "N/A")
            std = num_stats.get("std", "N/A")
            print(f"  {name}: mean={mean}, std={std}")
        else:
            print(f"  {name}: (non-numerical or no stats)")
except Exception as e:
    print(f"  Could not load statistics: {e}")

print("\n=== Baseline Constraints (summary) ===")
try:
    constraints = baseline_job.suggested_constraints().body_dict
    for feature in constraints.get("features", []):
        name = feature.get("name", "unknown")
        dtype = feature.get("inferred_type", "unknown")
        completeness = feature.get("completeness", "N/A")
        print(f"  {name}: type={dtype}, completeness={completeness}")
except Exception as e:
    print(f"  Could not load constraints: {e}")

## STEP 7 -- Create a Monitoring Schedule

A monitoring schedule is a recurring job that:
1. Grabs the captured data since the last run
2. Compares feature distributions to the baseline
3. Produces a violations report listing features that exceeded thresholds

We use an hourly schedule for demonstration purposes. In production, daily or every-6-hours is typical.

> **What is happening:** We create a monitoring schedule that runs hourly, compares captured endpoint data to the baseline, and writes violation reports to S3.

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator

SCHEDULE_NAME = f"fraudshield-monitor-schedule-{TIMESTAMP}"
monitoring_output_s3 = f"s3://{bucket}/{prefix}/monitoring/reports/{TIMESTAMP}"

print(f"Creating monitoring schedule: {SCHEDULE_NAME}")
print(f"Reports will be written to: {monitoring_output_s3}")

monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME,
    endpoint_input=MONITOR_ENDPOINT,
    output_s3_uri=monitoring_output_s3,
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

print(f"\nMonitoring schedule created: {SCHEDULE_NAME}")
print("The first monitoring execution will run at the top of the next hour.")

schedule_desc = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName=SCHEDULE_NAME
)
print(f"Schedule status: {schedule_desc['MonitoringScheduleStatus']}")

## Data Quality vs Model Quality Monitoring

SageMaker provides two complementary monitoring types:

| Dimension | Data Quality Monitoring | Model Quality Monitoring |
|-----------|------------------------|------------------------|
| **What it monitors** | Input feature distributions | Prediction accuracy vs ground truth |
| **Baseline source** | Training data statistics | Metrics from a baseline evaluation |
| **Detects** | Data drift, missing values, type changes | Accuracy drop, precision/recall degradation |
| **Ground truth needed?** | No | Yes (must be provided separately) |
| **SageMaker class** | `DefaultModelMonitor` | `ModelQualityMonitor` |
| **Latency to detect** | Near-real-time (next schedule) | Delayed (ground truth arrives later) |

Data quality monitoring is the **canary**: it catches distributional shifts without waiting for ground truth labels. Model quality monitoring provides **definitive** accuracy degradation evidence but requires ground truth, which for fraud detection arrives with a delay (investigations take weeks).

**Discussion:** For FraudShield, how would you get ground truth labels for model quality monitoring? What is the typical delay, and how does that affect your monitoring strategy?

## STEP 8 -- Generate Capture Data

The monitoring schedule needs captured data to analyze. We invoke the endpoint with validation data to generate capture logs.

> **What is happening:** We send multiple rows of validation data through the monitored endpoint. Each invocation is logged by the DataCaptureConfig -- request payload, response payload, and timestamp are written to S3 in JSON Lines format.

In [ ]:
import numpy as np

print("Sending validation data through the monitored endpoint...")
predictions = []
batch_size = 10
num_batches = min(20, len(features) // batch_size)

for i in range(num_batches):
    batch = features.iloc[i * batch_size : (i + 1) * batch_size]
    payload = batch.to_csv(index=False, header=False)

    response = runtime_client.invoke_endpoint(
        EndpointName=MONITOR_ENDPOINT,
        ContentType="text/csv",
        Body=payload,
    )
    result = response["Body"].read().decode("utf-8").strip()
    batch_preds = [float(x) for x in result.split(",")]
    predictions.extend(batch_preds)

print(f"Sent {num_batches * batch_size} records through the endpoint.")
print(f"Collected {len(predictions)} predictions.")
print(f"Prediction distribution: mean={np.mean(predictions):.3f}, std={np.std(predictions):.3f}")

time.sleep(10)

capture_prefix = f"{prefix}/data-capture/{MONITOR_ENDPOINT}"
capture_count = 0
for page in paginator.paginate(Bucket=bucket, Prefix=capture_prefix):
    capture_count += len(page.get("Contents", []))

print(f"\nCapture files in S3: {capture_count}")
if capture_count > 0:
    print("Data capture is working. Monitoring schedule will analyze these files.")
else:
    print("No capture files yet. They may take a minute to appear in S3.")

---
# STAGE 3 -- Drift Detection and Automation

The monitoring schedule is running. But monitoring is only useful if you can interpret the results and act on them. In this stage, we simulate what happens when the data changes, understand how Model Monitor flags the change, and design an automated response.

## Types of Drift

Three types of drift degrade model performance over time:

| Drift Type | What Changes | Example | Detection Method |
|------------|-------------|---------|------------------|
| **Data Drift** | Input feature distributions shift | Transaction amounts increase due to inflation | Compare feature statistics to baseline |
| **Concept Drift** | Relationship between features and target changes | A previously safe merchant category becomes high-risk | Monitor prediction accuracy over time (requires ground truth) |
| **Prediction Drift** | Model output distribution shifts | Model suddenly predicts more fraud than usual | Track prediction distribution |

**Data drift is the canary.** You can detect it without ground truth labels. Concept drift requires ground truth, which arrives with a delay. If you catch data drift early, you can investigate and retrain before concept drift causes real damage.

**Discussion:** Which type of drift is hardest to detect? Why? (Concept drift -- the features may look the same, but the relationship to the target has changed. Without ground truth, you cannot tell.)

## STEP 9 -- Simulate Data Drift

We simulate a real-world scenario: three months after deployment, transaction amounts have increased by 40% due to inflation, and the merchant risk score distribution has shifted because of a new partner onboarding.

> **What is happening:** We modify the validation data to simulate drift -- increasing `amount` by 40% and shifting `merchant_risk_score` up by 0.2. Then we send this drifted data through the monitored endpoint. When the monitoring schedule runs, it will compare these shifted distributions to the training baseline and flag violations.

In [ ]:
import matplotlib.pyplot as plt

drifted_features = features.copy()
drifted_features["amount"] = drifted_features["amount"] * 1.4
drifted_features["merchant_risk_score"] = (drifted_features["merchant_risk_score"] + 0.2).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(features["amount"], bins=30, alpha=0.6, label="Original", color="steelblue")
axes[0].hist(drifted_features["amount"], bins=30, alpha=0.6, label="Drifted (+40%)", color="coral")
axes[0].set_title("Amount Distribution: Original vs Drifted")
axes[0].legend()
axes[0].set_xlabel("Amount")

axes[1].hist(features["merchant_risk_score"], bins=30, alpha=0.6, label="Original", color="steelblue")
axes[1].hist(drifted_features["merchant_risk_score"], bins=30, alpha=0.6, label="Drifted (+0.2)", color="coral")
axes[1].set_title("Merchant Risk Score: Original vs Drifted")
axes[1].legend()
axes[1].set_xlabel("Risk Score")

plt.tight_layout()
plt.show()

print("Original amount  -- mean: {:.2f}, std: {:.2f}".format(features["amount"].mean(), features["amount"].std()))
print("Drifted amount   -- mean: {:.2f}, std: {:.2f}".format(drifted_features["amount"].mean(), drifted_features["amount"].std()))
print("Original risk    -- mean: {:.3f}, std: {:.3f}".format(features["merchant_risk_score"].mean(), features["merchant_risk_score"].std()))
print("Drifted risk     -- mean: {:.3f}, std: {:.3f}".format(drifted_features["merchant_risk_score"].mean(), drifted_features["merchant_risk_score"].std()))

In [ ]:
print("Sending DRIFTED data through the monitored endpoint...")
drifted_predictions = []

for i in range(num_batches):
    batch = drifted_features.iloc[i * batch_size : (i + 1) * batch_size]
    payload = batch.to_csv(index=False, header=False)

    response = runtime_client.invoke_endpoint(
        EndpointName=MONITOR_ENDPOINT,
        ContentType="text/csv",
        Body=payload,
    )
    result = response["Body"].read().decode("utf-8").strip()
    batch_preds = [float(x) for x in result.split(",")]
    drifted_predictions.extend(batch_preds)

print(f"Sent {num_batches * batch_size} drifted records through the endpoint.")
print(f"\nPrediction comparison:")
print(f"  Original data -- mean prediction: {np.mean(predictions):.3f}")
print(f"  Drifted data  -- mean prediction: {np.mean(drifted_predictions):.3f}")
print(f"\nThe drifted data is now captured alongside the original data.")
print("When the monitoring schedule runs, it will compare the combined captured")
print("distributions against the training baseline and flag the drifted features.")

## Statistical Tests for Drift Detection

Model Monitor uses statistical tests to quantify whether a feature has drifted from its baseline distribution:

| Test | Used For | What It Measures | Null Hypothesis |
|------|----------|-----------------|------------------|
| **Kolmogorov-Smirnov (KS)** | Continuous features | Maximum distance between two cumulative distribution functions (CDFs) | The two distributions are identical |
| **Chi-Squared** | Categorical features | Divergence between observed and expected bin frequencies | No significant difference between distributions |
| **L-Infinity Norm** | Continuous or categorical | Maximum absolute difference in bin frequencies | The distributions are identical |

**How it works in practice:**

1. Model Monitor computes the test statistic for each feature comparing captured data to baseline.
2. It compares the statistic to the threshold in `constraints.json`.
3. If the statistic exceeds the threshold, the feature is flagged as a **violation**.

**Example interpretation:**

| Feature | KS Statistic | Threshold | Result |
|---------|-------------|-----------|--------|
| `amount` | 0.42 | 0.10 | VIOLATION -- distribution shifted significantly |
| `hour` | 0.03 | 0.10 | Within bounds -- no drift detected |
| `merchant_risk_score` | 0.31 | 0.10 | VIOLATION -- distribution shifted |

The violations report lists every feature that failed its statistical test, along with the test statistic, the threshold, and a human-readable description of the violation.

## Bias and Attribution Drift

Beyond feature distributions, two advanced drift types matter for production ML systems:

### Bias Drift

The model starts treating demographic groups differently over time.

- **Example:** FraudShield's fraud model becomes more aggressive toward transactions from certain geographic regions as the data distribution shifts.
- **Detection:** SageMaker Clarify monitors bias metrics (DPL, DI, etc.) over time.
- **Requirement:** Regulatory mandate in many financial services contexts.

### Feature Attribution Drift

The importance of features changes over time.

- **Example:** At training time, `merchant_risk_score` was the top predictor. After drift, `amount` becomes more important because its distribution shifted significantly.
- **Detection:** SageMaker Clarify computes SHAP values on incoming data and compares them to baseline attributions.
- **Implication:** If feature importance shifts significantly, the model's decision logic may no longer match the current data regime.

Clarify provides both bias monitoring and explainability monitoring as optional add-ons to Model Monitor. They require additional configuration and compute, but for regulated industries they are essential.

**Discussion:** Why would feature attribution drift concern a compliance team even if overall model accuracy has not changed? (The model may still be accurate on average but making decisions for different reasons -- potentially discriminatory or unexplainable reasons.)

## EventBridge Automation Pattern

Detection without action is just expensive logging. The real value of monitoring is **automated response**. EventBridge bridges the gap between "drift detected" and "do something about it."

### Architecture

```
Model Monitor Schedule
    |
    v
Violations Detected --> CloudWatch Metrics (violation count per feature)
    |
    v
CloudWatch Alarm (triggers when violations exceed threshold)
    |
    v
EventBridge Rule (matches alarm state change event)
    |
    v
Target: Lambda / Step Functions / SageMaker Pipeline
    |
    v
Automated Retraining Pipeline
    |
    v
New Model --> Registry --> Human Approval --> Deployment
```

### Key Components

| Component | Role | Example |
|-----------|------|---------|
| **CloudWatch Metrics** | Model Monitor emits metrics for each monitoring execution | `feature_baseline_drift_amount` |
| **CloudWatch Alarm** | Triggers when a metric crosses a threshold | "Alert when 3+ features violate constraints" |
| **EventBridge Rule** | Matches alarm state changes and routes events | Rule: alarm -> ALARM state -> invoke target |
| **Lambda** | Lightweight handler: send Slack alert, trigger pipeline | Parse alarm, post to #ml-alerts, start retrain |
| **SageMaker Pipeline** | Full retraining DAG | Preprocess -> Train -> Evaluate -> Register |

### Design Decision: Auto-deploy or Human Gate?

For FraudShield, financial impact is high. The recommended pattern:
- **Automate** everything up to the approval gate (data collection, retraining, evaluation, registration)
- **Require human approval** for deployment (a senior data scientist reviews the new model's metrics before it goes live)

This is the end-to-end MLOps loop: deploy, monitor, detect, retrain, approve, redeploy.

**Discussion:** Should automated retraining deploy the new model automatically, or should it require human approval? What factors influence this decision? (Risk tolerance, regulatory requirements, model criticality, team size, frequency of drift.)

## Mandatory Cleanup -- Delete Everything

Delete all resources created during this notebook: endpoints, monitoring schedules, model objects, and optionally the S3 capture and batch output data.

**Cleanup order:**
1. Stop and delete the monitoring schedule (it runs recurring jobs that cost money)
2. Delete the monitored real-time endpoint
3. Delete endpoint configurations
4. Delete model objects

> **What is happening:** We systematically delete every AWS resource created in this notebook. After running this cell, open the AWS Console, navigate to Inference > Endpoints, and verify the list is empty. Then check Billing.

In [ ]:
print("=== CLEANUP: Deleting all resources ===")
print()

print("1. Deleting monitoring schedule...")
try:
    monitor.delete_monitoring_schedule()
    print(f"   Deleted schedule: {SCHEDULE_NAME}")
except Exception as e:
    print(f"   Schedule: {e}")

print("\n2. Deleting monitored endpoint...")
safe_delete_endpoint(MONITOR_ENDPOINT)
safe_delete_model(f"fraudshield-xgb-monitor-{TIMESTAMP}")

print("\n3. Verifying Stage 1 resources are gone...")
for name in [SERVERLESS_ENDPOINT, ASYNC_ENDPOINT]:
    try:
        sm_client.describe_endpoint(EndpointName=name)
        print(f"   WARNING: {name} still exists! Deleting...")
        safe_delete_endpoint(name)
    except sm_client.exceptions.ClientError:
        print(f"   {name}: confirmed deleted")

for model_suffix in [f"fraudshield-xgb-{TIMESTAMP}",
                      f"fraudshield-xgb-async-{TIMESTAMP}",
                      f"fraudshield-xgb-batch-{TIMESTAMP}"]:
    try:
        sm_client.describe_model(ModelName=model_suffix)
        print(f"   WARNING: model {model_suffix} still exists! Deleting...")
        safe_delete_model(model_suffix)
    except sm_client.exceptions.ClientError:
        print(f"   Model {model_suffix}: confirmed deleted")

print("\n=== CLEANUP COMPLETE ===")
print("\nVERIFY MANUALLY:")
print("  1. Open AWS Console > SageMaker > Inference > Endpoints")
print("  2. Confirm no endpoints remain")
print("  3. Check Billing & Cost Management")

---
## Wrap-up

### What You Accomplished Today

1. **Inference Patterns:** You deployed Thursday's XGBoost model four different ways -- serverless (scale-to-zero, pay-per-invocation), asynchronous (large payloads, fire-and-forget), batch transform (bulk scoring, ephemeral infrastructure), and you understand multi-model endpoints conceptually. You now have a decision matrix to match any workload to the right inference pattern.

2. **Model Monitoring:** You deployed an endpoint with data capture, created a statistical baseline from training data, scheduled hourly data quality monitoring, and sent data through the endpoint to generate capture logs.

3. **Drift Detection:** You simulated data drift by shifting feature distributions, visualized the difference, and understood how statistical tests (KS, chi-squared, L-infinity) quantify drift. You also learned about bias drift and feature attribution drift.

4. **Automation:** You designed an EventBridge pattern that closes the loop from drift detection to automated retraining -- the complete MLOps feedback cycle.

### W3 Monday Preview

Next week shifts to advanced ML and generative AI. Monday introduces **foundation models**, **transfer learning at scale**, and **Amazon Bedrock**. The question changes from "how do we train and deploy our own models" to "how do we leverage pre-trained foundation models for our tasks." Read the Foundation Models and Bedrock concept threads before Monday.

### Key Takeaways

- Real-time is one of five inference patterns -- always start with the traffic pattern, not the instance type.
- A deployed model without monitoring is a liability, not an asset.
- Data drift is the canary for concept drift -- catch it early, investigate, retrain.
- Automate detection and retraining, but keep a human gate before production deployment.